# Import Data

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
import torch
import multiprocessing
import os


class CFG:
    
    target_name='tag'
    seed = 58
    
    #######################################################################################
    # GPU
    gpu_available = torch.cuda.is_available()

    print(f"CUDA available: {gpu_available}")
    if gpu_available:
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
    else:
        print(f'Use CPU, \nNumber of CPUs: {multiprocessing.cpu_count()}')
    
    #######################################################################################
    # Seed
    
    @staticmethod
    def seed_all(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CFG.seed_all(CFG.seed)

warnings.simplefilter('ignore')
sns.set_theme(style="ticks")

CUDA available: True
GPU name: Tesla T4
Number of GPUs: 2


In [2]:
x = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-2/x_train.csv')
y = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-2/y_train.csv')
x = x.drop('Unnamed: 0', axis=1)
y = y.drop('Unnamed: 0', axis=1)

# Prepare data

In [3]:
from sklearn.model_selection import train_test_split


x_train, x_val, y_train, y_val = train_test_split(
    x, y, stratify=y, shuffle=True, test_size=0.2, random_state=CFG.seed
)

# Optuna

In [4]:
import optuna

from catboost import CatBoostClassifier

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def objective(trial):

    if CFG.gpu_available:
        device = "GPU"
        n_jobs_setting = 1
    else:
        device = "CPU"
        n_jobs_setting = -1

    param_space = {
        'loss_function': 'Logloss',
        # 'grow_policy': 'SymmetricTree',
        'eval_metric': 'AUC',
        'use_best_model': True,
        'task_type': device,
        'border_count': 256,
        'verbose': 0,
        'random_seed': CFG.seed,
    
        'iterations': trial.suggest_int('iterations', 100, 3000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.001, 10.0, log=True),
        'model_size_reg': trial.suggest_float('model_size_reg', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 2.0),
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 200, step=10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
    }

    # Var 1
    # local_x_train, local_x_val, local_y_train, local_y_val = train_test_split(
    #     x, y, stratify=y, shuffle=True, test_size=0.25
    # )
    # clf = XGBClassifier(**params).fit(local_x_train, local_y_train, eval_set=[(local_x_val, local_y_val)], verbose=0)
    # y_pred_proba = clf.predict_proba(local_x_val)[:, 1]
    # auc_score = roc_auc_score(local_y_val, y_pred_proba)
    # return {'score': auc_score, 'status': STATUS_OK}
    
    # Var 2
    seed = param_space['random_seed'] # solve ModuleNotFoundError
    
    # model = CatBoostClassifier(**params)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    split_generator = skf.split(x_train, y_train)
    
    train_scores = []
    val_scores = []
    for fold, (train_idx, val_idx) in enumerate(split_generator):
        
        fold_x_train = x_train.iloc[train_idx, :]
        fold_y_train = y_train.iloc[train_idx]
        fold_x_val = x_train.iloc[val_idx, :]
        fold_y_val = y_train.iloc[val_idx]
        
        fold_model = CatBoostClassifier(**param_space)
        
        fold_model.fit(
            fold_x_train,
            fold_y_train,
            eval_set=(fold_x_val, fold_y_val),
            plot=False
        )
        
        fold_train_pred_proba = fold_model.predict_proba(fold_x_train)[:, 1]
        fold_val_pred_proba = fold_model.predict_proba(fold_x_val)[:, 1]
        
        fold_train_score = roc_auc_score(fold_y_train, fold_train_pred_proba)
        fold_val_score = roc_auc_score(fold_y_val, fold_val_pred_proba)
        
        train_scores.append(fold_train_score)
        val_scores.append(fold_val_score)

    train_scores = np.array(train_scores)
    val_scores = np.array(val_scores)

    trial.set_user_attr("train_scores_mean", train_scores.mean())

    # return val_scores.mean() - 4 * abs(val_scores.mean() - train_scores.mean())
    return val_scores.mean()


study = optuna.create_study(
    direction='maximize', 
    study_name='catboost_optimization',
    load_if_exists=True
)

study.optimize(objective, n_trials=600, timeout=2.5*3600, n_jobs=1, show_progress_bar=True)

[I 2026-02-28 06:54:54,914] A new study created in memory with name: catboost_optimization


  0%|          | 0/600 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-02-28 06:56:05,774] Trial 0 finished with value: 0.950170172403576 and parameters: {'iterations': 600, 'learning_rate': 0.005029147140981094, 'depth': 9, 'l2_leaf_reg': 3.8156072385863684, 'model_size_reg': 0.15799012609520968, 'random_strength': 1.5091680208946154, 'bagging_temperature': 1.371036764123941, 'early_stopping_rounds': 40, 'min_data_in_leaf': 9}. Best is trial 0 with value: 0.950170172403576.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


...

[I 2026-02-28 09:21:48,811] Trial 93 finished with value: 0.9565891624717144 and parameters: {'iterations': 2700, 'learning_rate': 0.015457364481720135, 'depth': 5, 'l2_leaf_reg': 6.711573063347574, 'model_size_reg': 0.19883126543461252, 'random_strength': 1.758018784953233, 'bagging_temperature': 1.0990371577924334, 'early_stopping_rounds': 170, 'min_data_in_leaf': 3}. Best is trial 73 with value: 0.9572242043805593.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-02-28 09:23:48,355] Trial 94 finished with value: 0.9572061833462481 and parameters: {'iterations': 2950, 'learning_rate': 0.016673065420993834, 'depth': 6, 'l2_leaf_reg': 5.176932628780984, 'model_size_reg': 0.25028205468108294, 'random_strength': 1.8384427939065024, 'bagging_temperature': 0.2017449257992011, 'early_stopping_rounds': 200, 'min_data_in_leaf': 6}. Best is trial 73 with value: 0.9572242043805593.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-02-28 09:25:55,846] Trial 95 finished with value: 0.9563183208735733 and parameters: {'iterations': 2800, 'learning_rate': 0.016928190813132786, 'depth': 6, 'l2_leaf_reg': 9.974704774760903, 'model_size_reg': 0.25686171759662846, 'random_strength': 1.8874750651157386, 'bagging_temperature': 1.6161375904521407, 'early_stopping_rounds': 200, 'min_data_in_leaf': 30}. Best is trial 73 with value: 0.9572242043805593.


In [5]:
best_trial = study.best_trial
print(f"Best validation score: {best_trial.value}")
print(f"Best train score: {best_trial.user_attrs['train_scores_mean']}")

best_params = study.best_params
print("Best parameters:", best_params)

Best validation score: 0.9572242043805593
Best train score: 0.9725899184284488
Best parameters: {'iterations': 2700, 'learning_rate': 0.01574302484149115, 'depth': 6, 'l2_leaf_reg': 6.1215989471844665, 'model_size_reg': 0.32272425450156916, 'random_strength': 1.947098105494967, 'bagging_temperature': 0.27087996778188433, 'early_stopping_rounds': 170, 'min_data_in_leaf': 9}


In [6]:
import plotly.graph_objects as go


trials_df = study.trials_dataframe()
val_scores = trials_df['value'].values
train_scores = [t.user_attrs['train_scores_mean'] for t in study.trials]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(val_scores))),
    y=val_scores,
    mode='markers+lines',
    name='Validation Score',
    marker={'color': 'blue'}
))

fig.add_trace(go.Scatter(
    x=list(range(len(train_scores))),
    y=train_scores,
    mode='markers+lines',
    name='Train Score',
    marker={'color': 'red'}
))

fig.update_layout(
    title='Optimization History - Train vs Validation Scores',
    xaxis_title='Trial',
    yaxis_title='AUC Score',
    hovermode='x unified'
)

fig.write_html("optimization_history.html")
fig.show()